# Lab: Solving FrozenLake with Value Iteration
By the end of this lab, you should be able to:
1. Inspect the known transition model of a Gymnasium environment.
2. Connect `env.unwrapped.P` to the MDP quantities $p(s',r\mid s,a)$.
3. Implement the Value Iteration algorithm.

> Value Iteration assumes that the environment model is known.

## 0. Install and import the required libraries

Run the installation cell in Google Colab. If Gymnasium is already installed, the command will finish quickly.

In [76]:
# Run this cell in Google Colab or Jupyter if Gymnasium is not installed.
%pip install -q gymnasium 

Note: you may need to restart the kernel to use updated packages.


In [77]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## 1. Introduction to ~OpenAI gym~ Gymnasium
In this notebook we will be using [gymnasium](https://github.com/Farama-Foundation/Gymnasium), a great toolkit for developing and comparing Reinforcement Learning algorithms. It provides many environments for your learning *agents* to interact with. 

Gymasium: https://gymnasium.farama.org/index.html 

**Tip**: `gym.envs.registry` is a dictionary containing all available environments

In [78]:
envs = gym.envs.registry
sorted(envs.keys())[:15] + ["..."]

['Acrobot-v1',
 'Ant-v2',
 'Ant-v3',
 'Ant-v4',
 'Ant-v5',
 'BipedalWalker-v3',
 'BipedalWalkerHardcore-v3',
 'Blackjack-v1',
 'CarRacing-v3',
 'CartPole-v0',
 'CartPole-v1',
 'CliffWalking-v1',
 'CliffWalkingSlippery-v1',
 'FrozenLake-v1',
 'FrozenLake8x8-v1',
 '...']

## 2. Create the FrozenLake environment

We use the standard $4\times4$ FrozenLake map.

- `S`: start
- `F`: frozen surface
- `H`: hole
- `G`: goal

When `is_slippery=True`, the action selected by the agent may not lead exactly in the intended direction. This makes the transition model stochastic.

States:
```text
 0(S)   1     2     3
 4      5(H)  6     7(H)
 8      9     10(H) 11
12(H)  13     14    15(G)
```

In [79]:
env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False
)
#env = gym.make("FrozenLake-v1", is_slippery=False, render_mode="human")
#shows the specification for the CartPole-v1 environment



# Action names for readability.
action_names = {
    0: "Left",
    1: "Down",
    2: "Right",
    3: "Up"
}
possible_actions = [np.arange(env.action_space.n) 
                    for state in range(env.observation_space.n)]


print("Number of states:", env.observation_space.n)
print("Number of actions:", env.action_space.n)

print()
for action_id, action_name in action_names.items():
    print(action_id, "=", action_name)

print()
print("Map:")
print(env.unwrapped.desc.astype(str))

Number of states: 16
Number of actions: 4

0 = Left
1 = Down
2 = Right
3 = Up

Map:
[['S' 'F' 'F' 'F']
 ['F' 'H' 'F' 'H']
 ['F' 'F' 'F' 'H']
 ['H' 'F' 'F' 'G']]



<span style="color:blue">
### Question 1

According to the output, how many states and actions are in this environment?
</span>

## 3. Inspect the known transition model

For FrozenLake, Gymnasium exposes the full model through: $P[state][action]$, which returns all possible outcomes after taking the selected action from the selected state.

```python
env.unwrapped.P[state][action]
```

Each returned tuple has the form:
```text
(probability, next_state, reward, terminated)
```

The correspondence is:
- `probability` $\rightarrow p(s',r\mid s,a)$, or more informally the probability of the listed transition.
- `next_state` $\rightarrow s'$.
- `reward` $\rightarrow r$.
- `terminated` indicates whether $s'$ is terminal.

Because the model lists every possible transition and its probability, this is a **known-model** or **model-based** setting.

For example, if the state=0, action =1
```text
Possible outcomes:
Left  → wall → state 0 → reward 0
Down         → state 4  → reward 0
Right        → state 1  → reward 0
```

In [80]:
state = 0
action = 1  # Down

transitions = env.unwrapped.P[state][action]

print(f"Transitions from state {state} after action {action}:")
for probability, next_state, reward, terminated in transitions:
    print(
        f"probability={probability:.3f}, " #a floating-point number with exactly 3 digits after the decimal point.
        f"next_state={next_state}, "
        f"reward={reward}, "
        f"terminated={terminated}"
    )

Transitions from state 0 after action 1:
probability=1.000, next_state=4, reward=0, terminated=False


<span style="color:blue">
### Question 2

What happens when you change `is_slippery=True` to `is_slippery=False`?  Does each state–action pair still have multiple possible next states?
</span> 

## 4. Compute one *Action Value*

For Value Iteration, the action value at iteration $k$ is:

$ Q_k(s,a) = \sum_{s',r} p(s',r\mid s,a) \left[r+\gamma V_k(s')\right]$

For a terminal transition, there is no future value after termination, so we use:

$r+\gamma\cdot 0$


In [81]:
def calculate_action_value(env, state, action, V, gamma):
    """Calculate Q_k(s, a) from the known transition model."""
    action_value = 0.0

    for probability, next_state, reward, terminated in env.unwrapped.P[state][action]:
        future_value = 0.0 if terminated else V[next_state]

        action_value += probability * (reward + gamma * future_value)

    return action_value

Below code calculates the Q-value of every possible action at $state=0$, using an initial state-value table where all values are zero.

In [82]:
n_states = env.observation_space.n #total number of states = 16
n_actions = env.action_space.n #total number of actions = 4

V = np.zeros(n_states)
gamma = 0.9

state = 0

print(f"Action values at state {state}, using the initial V(s)=0:")
for action in range(n_actions):
    q_value = calculate_action_value(
        env=env,
        state=state,
        action=action,
        V=V,
        gamma=gamma
    )
    print(f"Action {action}: Q({state}, {action}) = {q_value:.3f}")

Action values at state 0, using the initial V(s)=0:
Action 0: Q(0, 0) = 0.000
Action 1: Q(0, 1) = 0.000
Action 2: Q(0, 2) = 0.000
Action 3: Q(0, 3) = 0.000


<span style="color:blue">
### Question 3

Why are many action values initially equal to zero? What about for state=14?
</span> 

## 5. Perform one full update (*Q-table*)

For each state:

$ V_{k+1}(s) = \max_a Q_k(s,a)$

The function below performs one complete sweep over all states.

In [83]:
def one_value_iteration(env, V, gamma):
    n_states = env.observation_space.n
    n_actions = env.action_space.n

    new_V = np.zeros(n_states)

    for state in range(n_states):
        action_values = [
            calculate_action_value(
                env=env,
                state=state,
                action=action,
                V=V,
                gamma=gamma
            )
            for action in range(n_actions)
        ]

        new_V[state] = max(action_values)

    return new_V

In [84]:
V0 = np.zeros(n_states)
V1 = one_value_iteration(env, V0, gamma=0.9)

print("V0:")
print(V0.reshape(4, 4))

print()
print("V1 after one full sweep:")
print(V1.reshape(4, 4))

V0:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

V1 after one full sweep:
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 1. 0.]]


<span style="color:blue">
### Question 4

Which states become positive first?
</span> 

## 6. Implement the complete value iteration
*Update the Q-Table and Extract the Optimal Policy*

We repeatedly apply the Bellman optimality update until the largest change is below a threshold:

$\Delta=\max_{s\in\mathcal S}\left|V_{k+1}(s)-V_k(s)\right|$

We stop when: $\Delta < \theta$

After Value Iteration converges, choose the action with the largest action value:

$\pi^*(s)\in\arg\max_a Q^*(s,a)$


In [85]:
def value_iteration(env,gamma=0.9,threshold=1e-8, max_iterations=10_000):

    """Run Value Iteration and return V, policy, and Q-table."""
    n_states = env.observation_space.n
    n_actions = env.action_space.n

    V = np.zeros(n_states)
    deltas = []

    # Value Iteration
    for iteration in range(max_iterations):
        q_table = np.zeros((n_states, n_actions))
        # Calculate Q_k(s, a) for all states and actions
        for state in range(n_states):
            for action in range(n_actions):
                q_table[state, action] = calculate_action_value(
                    env=env,
                    state=state,
                    action=action,
                    V=V,
                    gamma=gamma
                )
        #V_(k+1)(s)=max_a Q_k(s, a)
        new_V = np.max(q_table, axis=1)

        delta = np.max(np.abs(new_V - V))
        deltas.append(delta)

        V=new_V

        if delta < threshold:
            break

    # Extract greedy policy
    final_q_table = np.zeros((n_states, n_actions))

    for state in range(n_states):
        for action in range(n_actions):
            final_q_table[state, action] = calculate_action_value(
                env=env,
                state=state,
                action=action,
                V=V,
                gamma=gamma
            )

    policy = np.argmax(final_q_table, axis=1)

    return V, policy, final_q_table, iteration + 1, deltas


**Note:**

`np.argmax` returns the index of the first largest value. 

If multiple actions tie, `np.argmax` chooses the first one. Therefore, another equally optimal policy may exist even if it is not displayed.

### Test the function

In [86]:
V, policy, q_table, iterations, deltas = value_iteration(
    env,
    gamma=0.9,
    threshold=1e-8
)

print(f"Converged after {iterations} iterations.")

print("\nOptimal state values:")
print(np.round(V.reshape(4, 4), 3))

print("\nOptimal policy as action numbers:")
print(policy.reshape(4, 4))

print("\nFinal Q-table:")
print(np.round(q_table,3))


Converged after 7 iterations.

Optimal state values:
[[0.59  0.656 0.729 0.656]
 [0.656 0.    0.81  0.   ]
 [0.729 0.81  0.9   0.   ]
 [0.    0.9   1.    0.   ]]

Optimal policy as action numbers:
[[1 2 1 0]
 [1 0 1 0]
 [2 1 1 0]
 [0 2 2 0]]

Final Q-table:
[[0.531 0.59  0.59  0.531]
 [0.531 0.    0.656 0.59 ]
 [0.59  0.729 0.59  0.656]
 [0.656 0.    0.59  0.59 ]
 [0.59  0.656 0.    0.531]
 [0.    0.    0.    0.   ]
 [0.    0.81  0.    0.656]
 [0.    0.    0.    0.   ]
 [0.656 0.    0.729 0.59 ]
 [0.656 0.81  0.81  0.   ]
 [0.729 0.9   0.    0.729]
 [0.    0.    0.    0.   ]
 [0.    0.    0.    0.   ]
 [0.    0.81  0.9   0.729]
 [0.81  0.9   1.    0.81 ]
 [0.    0.    0.    0.   ]]


<span style="color:blue">
### Question 5

1. Why do we use the maximum change across all states? Why not average change?
2. What should happen to $\Delta$? </span> 

### Test the optimal policy

In [87]:
# Convert action numbers into readable action names.
policy_names = np.array(
    [action_names[action] for action in policy]
)

# Reshape the policy into the same shape as the FrozenLake map.
policy_grid = policy_names.reshape(env.unwrapped.desc.shape)

print("Optimal policy:")
print(policy_grid)

Optimal policy:
[['Down' 'Right' 'Down' 'Left']
 ['Down' 'Left' 'Down' 'Left']
 ['Right' 'Down' 'Down' 'Left']
 ['Left' 'Right' 'Right' 'Left']]


### Test the policy in the environment
Evaluate a policy over multiple episodes

In [88]:
episodes = 1_000
max_steps = 100
seed = 42

returns = []
steps_per_episode = []

for episode in range(episodes):
    state, _ = env.reset(seed=seed + episode)
    total_reward = 0.0

    for step in range(max_steps):
        action = int(policy[state])

        state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward

        if terminated or truncated:
            break

    returns.append(total_reward)
    steps_per_episode.append(step + 1)

success_rate = np.mean(np.array(returns) > 0)
mean_return = np.mean(returns)
mean_steps = np.mean(steps_per_episode)

print(f"Success rate: {success_rate:.1%}")
print(f"Mean return: {mean_return:.3f}")
print(f"Mean steps: {mean_steps:.2f}")

Success rate: 100.0%
Mean return: 1.000
Mean steps: 6.00



<span style="color:blue">
### Question 6

1. When you set `is_slippery=False`, the success rate is 100%. 
2. What happens when you set `is_slippery=True`?
3. Why might the success rate be below 100% even when the learned policy is optimal?
